# S. aureus / NR4A — ZWO camera

**Makes up Figure(s):** S75 (`ZWO_SAureus_Replicate`, section 9; the per-pixel figure in section 11).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/ZWO/20260624_SAureus_NR4A/FOV3/`
— *S. aureus* labelled with NR4A, imaged on the ZWO (ASI585MC) camera. The acquisition's
MicroManager `*_metadata.txt` (~0.95 GB) is always downloaded too: it holds the camera ROI
the frames were cropped to, which the fit needs to crop the calibration maps to match.

## FAST_MODE

Each raw acquisition file is a roughly 4.3 GB multi-frame OME-TIF; the full FOV3 acquisition is
27 such files roughly 230 GB). `FAST_MODE = True` (default) downloads and fits only the first
file (4.3 GB, 260 frames) plus the metadata — enough to run the whole pipeline end-to-end and see
sensible-looking output, but with far less signal than the published numbers. Set
`FAST_MODE = False` to download and process the complete FOV3 acquisition (**230 GB,
hours of download + fitting** — run in `tmux`/`screen`).

---
## 0 — Imports


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import pandas as pd
import seaborn as sns

# _biostudies_download.py is shared across notebooks/figures/ and its subfolders -- it lives
# one level up from here (notebooks/figures/), not beside this notebook (notebooks/figures/SI/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _nile_red as nr

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import MaskFunctions
from pyS3M import SpotDetectionFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import postprocess as _postprocess
from pyS3M import NileRedFunctions
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)


---
## 1 — Configuration


In [ ]:
FAST_MODE = True   # True: 1 file (~4.3 GB, illustrative). False: full FOV3 (~230 GB, real numbers).

# ── ZWO camera: pixel size (nm) and spectral/mask/spot-detection setup ───────────────────
PIXEL_SIZE_NM = 71.5   # ZWO (ASI585MC): 0.0715 um -- see src/CameraDefaults.py.
                        # (An earlier draft of the post-analysis logic this notebook is based
                        # on hardcoded 69 nm, the Ximea value -- this is the ZWO camera.)

S_F  = SpectralFunctions.Spectral_Funcs(camera="zwo")
M_F  = MaskFunctions.Mask_Functions(camera="zwo")
SD_F = SpotDetectionFunctions.SpotDetection_Functions(camera="zwo")

# camera="zwo" alone only sets SuperRes_Functions' own pixel_size/mosaic_unit -- its
# internal default mask_functions/spot_detection_functions sub-instances would otherwise
# silently stay ximea-configured (they don't inherit the camera argument), so inject the
# already-zwo-configured M_F/SD_F explicitly (same pattern the raw-analysis dev notebook
# used).
SupRes_F = SR_Functions.SuperRes_Functions(camera="zwo", mask_functions=M_F, spot_detection_functions=SD_F)

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])
camera_params = {"pixel_QYs": pixel_QYs, "wavelength": wavelength}

import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

# ── Paths ──────────────────────────────────────────────────────────────────────────────
# This notebook lives at notebooks/figures/SI/ -- same depth as the other ported FigureSI_*
# notebooks, so REPO_ROOT resolves the same way they do (e.g. FigureSI_XimeavsPrime95B.ipynb).
REPO_ROOT = Path.cwd().resolve().parents[2]
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "ZWO_Camera"
# Point DATA_DIR at a drive with room for the download (FAST_MODE ~5.3 GB, full ~230 GB).
# Keep it for this acquisition only: the fit reads every TIFF and metadata file inside it.
DATA_DIR  = Path("outputs") / "SAureus_ZWO_FOV3"   # git-ignored (bare outputs/ rule)
FIG_DIR   = Path("figures")                         # git-ignored (*.svg)
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

gain_map    = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map  = IO.read_tiff(CALIB_DIR / "offset.tif")
variance    = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise  = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe         = IO.read_tiff(CALIB_DIR / "rqe.tif")

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Calibration: {CALIB_DIR}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")


---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Lists the FOV3/Slide3_1 folder on the deposition and downloads either just the first
TIFF (`FAST_MODE`) or the complete acquisition, plus its `*_metadata.txt` in both cases
(`_biostudies_download.download_acquisition`). Re-running this cell is safe: files
already downloaded at the correct size are skipped, and interrupted ones resume.

In [ ]:
REMOTE_FOLDER = (
    "2026_Multicolour_Paper/ZWO/20260624_SAureus_NR4A/"
    "FOV3/500pM_NR4A_60perc_515_515LP650200BP_Slide3_1"
)
STEM = "500pM_NR4A_60perc_515_515LP650200BP_Slide3_1_MMStack_Pos0"

# The full acquisition is 27 TIFFs (the base file plus _1 .. _26) and one metadata file.
downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, stem=STEM, first_file_only=FAST_MODE)
image_folder = DATA_DIR

---
## 3 — Raw fit: spot detection + fitting across the downloaded frames

Same call as the raw-analysis dev notebook (`SuperRes_Functions.fit_imaging_data`,
already current-API-compatible — no signature changes needed there). Writes
`Localisations.h5` into `image_folder`.

This can take a while even in `FAST_MODE` (one ~4.3 GB file); several hours in full mode.


In [ ]:
pfa = 1e-3
peak_wavelength = 0.65

# Quick single-frame sanity check before committing to the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    image_folder, pfa=pfa, frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=peak_wavelength,
)
plt.show()


In [ ]:
SupRes_F.fit_imaging_data(
    image_folder,
    smoothing_function,
    gain_map,
    offset_map,
    rqe,
    read_noise,
    variance=variance,
    pfa=pfa,
    ROI_size=12,
    peak_wavelength=peak_wavelength,
    NA=1.49,
    # pixel_size intentionally omitted: defaults to SupRes_F.pixel_size (zwo -> 71.5 nm).
    image_type=".tif",
)


---
## 4 — Load fit results and apply quality-control filters

`width`/`height` are read from the first downloaded frame (`IO.read_tiff`); they equal the
ROI in the metadata file.

In [ ]:
localisation_files = H_F.file_search(image_folder, ".h5", "")
print(localisation_files)

first_frame = IO.read_tiff(downloaded["images"][0], frame=0)
height, width = first_frame.shape
print(f"Frame shape: {width} x {height} px")

loc_data = pd.read_hdf(localisation_files[0])
print(f"Raw localisations: {len(loc_data):,}")

plt.hist(loc_data["chi_sqr"], 100)
plt.xlim([0, 3])
plt.show()


In [ ]:
# Author's original QC cuts for this dataset -- copied verbatim from the dev notebook.
loc_data = loc_data[loc_data["xc_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 50 / PIXEL_SIZE_NM]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 50 / PIXEL_SIZE_NM]

loc_data = loc_data[(loc_data["s_x"] > 80 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 80 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.4]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

# published: < 500 `photons` (A_B + A_G + A_R); x 0.32 = 0.25 (1 + A_G), Nile Red's A_G ~0.3, on the Bayer mosaic
loc_data = loc_data[~((loc_data["photoelectrons"] < 160) & (loc_data["spot_matched_filter_response"] < 50))]
loc_data = loc_data[loc_data["spot_background_std"] > 10]
loc_data = loc_data[loc_data["chi_sqr"] < 2]

print(f"After QC filtering: {len(loc_data):,} localisations")

plt.hist(loc_data["A_R"], 1000, color="red")
plt.hist(loc_data["A_G"], 1000, alpha=0.5, color="darkgreen")
plt.xlim([0, 1])
plt.show()


---
## 5 — Drift correction (AIM) and frame-to-frame linking


In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=50,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM)
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM)
plt.show()


In [ ]:
# n_frames from the fitted data itself, so FAST_MODE (first file only) gets the right count.
n_frames = int(loc_data["frame"].max()) + 1

link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

IO.write_h5_database(df=linked, filepath=localisation_files[0].split(".h5")[0] + "_Undrifted_Linked.h5",
                      normalise_photons=False)


---
## 6 — Render the linked super-resolution image


In [ ]:
oversampling = 8
min_blur_width = 4 * (PIXEL_SIZE_NM / 1000)

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(linked["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

n_locs, image_FOV = render.render(
    locs=linked.to_records(index=False),
    info=info,
    blur_method="smooth",
    oversampling=oversampling,
    min_blur_width=min_blur_width,
)[:2]

fig, axs = plotter.two_column_plot(height=2, width=2.25)
axs = plotter.image_plot(axs, image_FOV, cmap="hot", vmax=np.percentile(image_FOV, 99.9), colorbar=False)

plt.savefig(FIG_DIR / "Image_SAureus_ZWO.svg", dpi=600, format="svg")
plt.show()


---
## 7 — Segment individual bacteria from the rendered image

`postprocess.segment_locs_by_rendered_image` — image-thresholding-based aggregate
detection, much cheaper than DBSCAN for this many localisations.


In [ ]:
MIN_AREA_NM2 = 100.0
MIN_LOCS = 20

file = localisation_files[0].split(".h5")[0] + "_Undrifted_Linked.h5"

bacteria_locs, per_bacteria_stats = _postprocess.segment_locs_by_rendered_image(
    linked,
    width=width,
    height=height,
    oversampling=oversampling,
    pixel_size_nm=PIXEL_SIZE_NM,
    min_area_nm2=MIN_AREA_NM2,
    min_localisations=MIN_LOCS,
    threshold_method="li",
    callback="console",
    blur_method="gaussian",
    verbose=True,
)

bacteria_locs = bacteria_locs.rename(columns={"aggregate_id": "cluster_id", "aggregate_area_nm2": "cluster_area_nm2"})
per_bacteria_stats = per_bacteria_stats.rename(columns={"aggregate_id": "cluster_id"})

IO.write_h5_database(df=bacteria_locs, filepath=file.split(".h5")[0] + "_clusteredlocs.h5",
                      append=False, normalise_photons=False)
IO.write_h5_database(df=per_bacteria_stats, filepath=file.split(".h5")[0] + "_averagedoverclusters.h5",
                      append=False, normalise_photons=False)

print(f"Detected {bacteria_locs['cluster_id'].nunique()} bacteria")


---
## 8 — Nile Red pixelated wavelength fit


In [ ]:
nrf = NileRedFunctions.NileRed_Functions()

filters = [
    "semrock-ff01-650-200",
    "semrock-di03-r514-t1-25x36",
    "semrock-ff01-515-lp",
]

analysis_files = H_F.file_search(image_folder, "_clusteredlocs.h5", "")
MIN_LOCS_ANALYSIS = 10

for f in analysis_files:
    df_with_wavelengths, grid_info = nrf.fit_wavelengths_pixelated(
        h5_path=f,
        filter_names=filters,
        camera_parameters=camera_params,   # zwo-configured (see "What changed" above)
        output_path=f,
        min_localisations=MIN_LOCS_ANALYSIS,
        pixel_size_nm=50,
        camera_pixel_size=PIXEL_SIZE_NM,   # ZWO 71.5 nm; the default is the Ximea's 69 nm
        aggregate_id_column="cluster_id",
    )

df_with_wavelengths = pd.read_hdf(analysis_files[0])
print(f"Wavelength-fitted rows: {len(df_with_wavelengths):,}")


---
## 9 — Figure S75: super-resolved and relative pixel 3 QE renders

a) the super-resolved render, b) each 50 nm pixel's relative pixel 3 QE $f_R = R/(R+G+B)$ of its summed
photoelectrons (pixels with fewer than 10 localisations take their bacterium's value), hue on the population
mean $\pm$ 2.5 SD over pixels, brightness = localisation density: the same render as Figure 5h
(`_nile_red.render_red_share`). Each panel has an inset of the same two cells. Drawn at the published size
(`_nile_red.replicate_figure`). `CROP` and `ZOOM` are in camera px of the full acquisition (the published field
and pair); in `FAST_MODE` the field is sparse.

In [ ]:
CROP = dict(x0=75, y0=55, w=940, h=850)   # camera px: the field shown (as published)
ZOOM = dict(x0=570, y0=430, size=36)       # camera px: the inset's two cells (as published)

# relative pixel 3 QE per 50 nm pixel, on the wavelength fit's grid (section 8)
rs_locs, rs_pixels = nr.red_share_pixelated(df_with_wavelengths, pixel_size_nm=50.0, camera_pixel_size_nm=PIXEL_SIZE_NM)
rs_mean, rs_std, rs_min, rs_max = nr.colour_scale(rs_pixels)
print(f"{len(rs_pixels):,} pixels: relative pixel 3 QE {rs_mean:.3f} +/- {rs_std:.3f} (SD); "
      f"median per-pixel error {rs_pixels['red_share_err'].median():.3f}")

# both panels on one grid: the whole camera ROI
viewport = ((0, 0), (height, width))
image_sr = render.render(locs=linked.to_records(index=False), oversampling=oversampling, viewport=viewport,
                         blur_method="smooth", min_blur_width=min_blur_width)[1]
_, image_rs = nr.render_red_share(render, rs_locs, rs_min, rs_max, oversampling, viewport=viewport)

fig, axs = nr.replicate_figure(image_sr, image_rs, oversampling, PIXEL_SIZE_NM, CROP, ZOOM,
                               dict(c_min=rs_min, c_max=rs_max, label=nr.PIXEL3_LABEL))
plt.savefig(FIG_DIR / "ZWO_SAureus_Replicate.svg", dpi=600, format="svg")
plt.show()

---
## 10 — Two-region spectral comparison

`rects` are the original author's hand-picked pixel regions for FOV3, copied verbatim —
**these will very likely need re-picking against the real rendered image** once run for
real, since they were tuned against the author's own already-downloaded data.


In [ ]:
def get_region_localisations(df, rectangles):
    """Filter localisations within each rectangle (camera pixels)."""
    regions = []
    for rect in rectangles:
        mask = (
            (df["xc"] >= rect["x0"]) & (df["xc"] < rect["x0"] + rect["w"]) &
            (df["yc"] >= rect["y0"]) & (df["yc"] < rect["y0"] + rect["h"])
        )
        regions.append(df[mask].copy())
    return regions


# Author-selected regions (absolute camera pixels) -- see markdown note above.
rects = [
    dict(x0=351, y0=425, w=4, h=5, color="#4363d8", label="Region A"),
    dict(x0=355, y0=424, w=4, h=5, color="#e6194b", label="Region B"),
]

region_A, region_B = get_region_localisations(df_with_wavelengths, rects)
print(f"Region A: {len(region_A)} localisations")
print(f"Region B: {len(region_B)} localisations")


In [ ]:
fig, axs = plotter.one_column_plot(width=2, height=1.5)

axs = plotter.histogram_plot(axs, region_A["A_R"], np.histogram_bin_edges(region_A["A_R"], "fd"),
                              density=False, color="#4363d8", alpha=0.5)
axs = plotter.histogram_plot(axs, region_B["A_R"], np.histogram_bin_edges(region_B["A_R"], "fd"),
                              density=False, color="#e6194b", alpha=0.5, xlabel="pixel 3 QE")

axs.vlines(ymin=0, ymax=100, x=np.mean(region_A["A_R"]), color="#4363d8", ls="--", lw=1)
axs.vlines(ymin=0, ymax=100, x=np.mean(region_B["A_R"]), color="#e6194b", ls="--", lw=1)

axs.set_ylim([0, 95])
plt.xlim([0.5, 0.9])
plt.savefig(FIG_DIR / "SAureus_Histogram.svg", dpi=600, format="svg")
plt.show()


---
## 11 — SI — per-pixel relative pixel 3 QE and wavelength (S75)

One value per 50 nm analysis pixel (bacterium, `pixel_ix`, `pixel_iy`) with at least 10 localisations (pixels
below that take their bacterium's value and are left out here; `pixel_ix` alone is only the grid's x column).
Top: relative pixel 3 QE; bottom: the fitted wavelength. Left: distribution, mean $\mu$ $\pm$ SD $\sigma$;
right: against log$_{10}$ of the pixel's summed detected photoelectrons. `_nile_red.per_pixel_table` checks that
the wavelength fit used this grid; `_nile_red.plot_per_pixel` draws it at the published panel's width.

In [ ]:
pixels = nr.per_pixel_table(df_with_wavelengths, camera_pixel_size_nm=PIXEL_SIZE_NM)
fig, axs, pixel_stats = nr.plot_per_pixel(pixels)
plt.savefig(FIG_DIR / "SAureus_Analysis_RelativeQE_Wavelength.svg", dpi=600, format="svg")
plt.show()
nr.print_per_pixel_stats(pixel_stats)